<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 8.4: Text Classification

In this lab you will implement different types of feature engineering for text classification:
* Count vectors
* TF-IDF vectors (word level, n-gram level, character level)
* Text/NLP based features
* Topic models
  
The following classification algorithms will be applied to the count and TF-IDF vector features:
* Naïve Bayes
* Logistic Regression
* Support Vector Machine
* Random Forest
* Gradient Boosting

## Import libraries

In [1]:
## Import Libraries
import numpy as np
import pandas as pd

import string
import spacy

from collections import Counter

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# import warnings
# warnings.filterwarnings('ignore')

## Load data

Sample:

    __label__2 Stuning even for the non-gamer: This sound ...
    __label__2 The best soundtrack ever to anything.: I'm ...
    __label__2 Amazing!: This soundtrack is my favorite m ...
    __label__2 Excellent Soundtrack: I truly like this so ...
    __label__2 Remember, Pull Your Jaw Off The Floor Afte ...
    __label__2 an absolute masterpiece: I am quite sure a ...
    __label__1 Buyer beware: This is a self-published boo ...
    . . .
    
There are only two **labels**:
- `__label__1`
- `__label__2`

In [2]:
from google.colab import files
uploaded = files.upload()

Saving corpus.txt to corpus.txt


In [21]:
## Loading the data

df_corpus = pd.read_fwf(
    filepath_or_buffer = 'corpus.txt',
    colspecs = [(9, 10),   # label: get only the numbers 1 or 2
                (11, 9000) # text: makes the it big enough to get to the end of the line
               ],
    header = 0,
    names = ['label', 'text'],
    lineterminator = '\n'
)

# convert label from [1, 2] to [0, 1]
df_corpus['label'] = df_corpus['label'] - 1

## Inspect the data

In [22]:
# ANSWER
df_corpus.head()

,label,text
0,1,The best soundtrack ever to anything.: I'm rea...
1,1,Amazing!: This soundtrack is my favorite music...
2,1,Excellent Soundtrack: I truly like this soundt...
3,1,"Remember, Pull Your Jaw Off The Floor After He..."
4,1,an absolute masterpiece: I am quite sure any o...


In [23]:
df_corpus.shape

(9999, 2)

## Split the data into train and test

In [24]:
## ANSWER
## split the dataset
# Features (texts) and labels
X = df_corpus['text']
y = df_corpus['label']

# 80 % train, 20 % test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")

Train size: 7,999  |  Test size: 2,000


## Feature Engineering

### Count Vectors as features

In [25]:
# create a count vectorizer object
count_vect = CountVectorizer(token_pattern = r'\w{1,}')

# Learn a vocabulary dictionary of all tokens in the raw documents
count_vect.fit(X_train)

# Transform documents to document-term matrix.
X_train_count = count_vect.transform(X_train)
X_test_count = count_vect.transform(X_test)

### TF-IDF Vectors as features
- Word level
- N-Gram level
- Character level

In [26]:
%%time
# word level tf-idf
tfidf_vect = TfidfVectorizer(analyzer = 'word',
                             token_pattern = r'\w{1,}',
                             max_features = 5000)
print(tfidf_vect)

tfidf_vect.fit(X_train)
X_train_tfidf = tfidf_vect.transform(X_train)
X_test_tfidf  = tfidf_vect.transform(X_test)

TfidfVectorizer(max_features=5000, token_pattern='\\w{1,}')
CPU times: user 1.82 s, sys: 3.95 ms, total: 1.82 s
Wall time: 1.88 s


In [27]:
%%time
# ngram level tf-idf
tfidf_vect_ngram = TfidfVectorizer(analyzer = 'word',
                                   token_pattern = r'\w{1,}',
                                   ngram_range = (2, 3),
                                   max_features = 5000)
print(tfidf_vect_ngram)

tfidf_vect_ngram.fit(X_train)
X_train_tfidf_ngram = tfidf_vect_ngram.transform(X_train)
X_test_tfidf_ngram  = tfidf_vect_ngram.transform(X_test)

TfidfVectorizer(max_features=5000, ngram_range=(2, 3), token_pattern='\\w{1,}')
CPU times: user 5.4 s, sys: 166 ms, total: 5.56 s
Wall time: 5.59 s


In [28]:
%%time
# characters level tf-idf
tfidf_vect_ngram_chars = TfidfVectorizer(analyzer = 'char',
                                         ngram_range = (2, 3),
                                         max_features = 5000)
print(tfidf_vect_ngram_chars)

tfidf_vect_ngram_chars.fit(X_train)
X_train_tfidf_ngram_chars = tfidf_vect_ngram_chars.transform(X_train)
X_test_tfidf_ngram_chars  = tfidf_vect_ngram_chars.transform(X_test)

TfidfVectorizer(analyzer='char', max_features=5000, ngram_range=(2, 3))
CPU times: user 8.29 s, sys: 35.9 ms, total: 8.33 s
Wall time: 8.42 s


### Text / NLP based features

Create some other features.

char_count = Number of Characters in Text

word_count = Number of Words in Text

word_density = Average Number of Char in Words

punctuation_count = Number of Punctuation in Text

title_word_count = Number of Words in Title

uppercase_word_count = Number of Upperwords in Text


In [29]:
%%time
# ANSWER
def extract_nlp_features(df):
    return pd.DataFrame({
        'char_count': df.apply(len),
        'word_count': df.apply(lambda x: len(x.split())),
        'word_density': df.apply(lambda x: len(x) / (len(x.split()) + 1e-5)),
        'punctuation_count': df.apply(lambda x: sum([1 for c in x if c in string.punctuation])),
        'title_word_count': df.apply(lambda x: sum([1 for w in x.split() if w.istitle()])),
        'uppercase_word_count': df.apply(lambda x: sum([1 for w in x.split() if w.isupper()])),
    })

# Apply to train and test
X_train_nlp = extract_nlp_features(X_train)
X_test_nlp  = extract_nlp_features(X_test)

# Preview
X_train_nlp.head()

CPU times: user 498 ms, sys: 1.01 ms, total: 499 ms
Wall time: 500 ms


,char_count,word_count,word_density,punctuation_count,title_word_count,uppercase_word_count
2310,259,49,5.285713,5,0,0
4599,464,80,5.799999,12,14,1
6979,819,134,6.111940,17,14,0
5420,248,44,5.636362,7,5,1
2342,1002,167,6.000000,27,9,2


In [30]:
## load spaCy
nlp = spacy.load('en_core_web_sm')

Part of Speech in **SpaCy**

    POS   DESCRIPTION               EXAMPLES
    ----- ------------------------- ---------------------------------------------
    ADJ   adjective                 big, old, green, incomprehensible, first
    ADP   adposition                in, to, during
    ADV   adverb                    very, tomorrow, down, where, there
    AUX   auxiliary                 is, has (done), will (do), should (do)
    CONJ  conjunction               and, or, but
    CCONJ coordinating conjunction  and, or, but
    DET   determiner                a, an, the
    INTJ  interjection              psst, ouch, bravo, hello
    NOUN  noun                      girl, cat, tree, air, beauty
    NUM   numeral                   1, 2017, one, seventy-seven, IV, MMXIV
    PART  particle                  's, not,
    PRON  pronoun                   I, you, he, she, myself, themselves, somebody
    PROPN proper noun               Mary, John, London, NATO, HBO
    PUNCT punctuation               ., (, ), ?
    SCONJ subordinating conjunction if, while, that
    SYM   symbol                    $, %, §, ©, +, −, ×, ÷, =, :), 😝
    VERB  verb                      run, runs, running, eat, ate, eating
    X     other                     sfpksdpsxmsa
    SPACE space
    
Find out the number of Adjectives, Adverbs, Nouns, Numerals, Pronouns, Proper Nouns, Verbs.

    Hint:
    1. Convert text to spacy document
    2. Use pos_
    3. Use Counter

In [31]:
# Initialise some columns for feature's counts
df_corpus['adj_count'] = 0
df_corpus['adv_count'] = 0
df_corpus['noun_count'] = 0
df_corpus['num_count'] = 0
df_corpus['pron_count'] = 0
df_corpus['propn_count'] = 0
df_corpus['verb_count'] = 0

In [36]:
# ANSWER
def pos_counts(text):
    doc = nlp(text)
    pos_counter = Counter([token.pos_ for token in doc])
    return {
        'adj_count': pos_counter.get('ADJ', 0),
        'adv_count': pos_counter.get('ADV', 0),
        'noun_count': pos_counter.get('NOUN', 0),
        'num_count': pos_counter.get('NUM', 0),
        'pron_count': pos_counter.get('PRON', 0),
        'propn_count': pos_counter.get('PROPN', 0),
        'verb_count': pos_counter.get('VERB', 0)
    }



# Convert to DataFrame and merge with df_corpus
pos_df = pd.DataFrame(pos_features.tolist())
df_corpus = pd.concat([df_corpus, pos_df], axis=1)

import string

def extract_text_features(df_text):
    return pd.DataFrame({
        'char_count': df_text.apply(len),
        'word_count': df_text.apply(lambda x: len(x.split())),
        'word_density': df_text.apply(lambda x: len(x) / (len(x.split()) + 1e-5)),
        'punctuation_count': df_text.apply(lambda x: sum([1 for c in x if c in string.punctuation])),
        'title_word_count': df_text.apply(lambda x: sum([1 for w in x.split() if w.istitle()])),
        'uppercase_word_count': df_text.apply(lambda x: sum([1 for w in x.split() if w.isupper()])),
    })

# Apply to the 'text' column in df_corpus
text_features_df = extract_text_features(df_corpus['text'])

# Add these features back into df_corpus
df_corpus = pd.concat([df_corpus, text_features_df], axis=1)

In [37]:
cols = [
    'char_count', 'word_count', 'word_density',
    'punctuation_count', 'title_word_count',
    'uppercase_word_count', 'adj_count',
    'adv_count', 'noun_count', 'num_count',
    'pron_count', 'propn_count', 'verb_count']

df_corpus[cols].sample(5)

,char_count,word_count,word_density,punctuation_count,title_word_count,uppercase_word_count,adj_count,adj_count,adv_count,adv_count,noun_count,noun_count,num_count,num_count,pron_count,pron_count,propn_count,propn_count,verb_count,verb_count
4544,797,155,5.141935,28,13,4,0,17,0,6,0,33,0,1,0,26,0,4,0,12
7556,210,35,5.999998,7,4,2,0,4,0,1,0,10,0,0,0,2,0,1,0,4
7796,217,37,5.864863,7,6,3,0,3,0,4,0,5,0,0,0,6,0,4,0,4
9211,931,167,5.574850,49,24,5,0,13,0,9,0,38,0,3,0,16,0,13,0,17
3888,615,113,5.442477,14,14,4,0,5,0,7,0,23,0,2,0,15,0,11,0,12


### Topic Models as features

In [38]:
%%time
# train a LDA Model
lda_model = LatentDirichletAllocation(n_components = 20, learning_method = 'online', max_iter = 20)

X_topics = lda_model.fit_transform(X_train_count)
topic_word = lda_model.components_
vocab = count_vect.get_feature_names_out()

CPU times: user 59 s, sys: 149 ms, total: 59.1 s
Wall time: 1min 1s


In [39]:
# view the topic models
n_top_words = 10
topic_summaries = []
print('Group Top Words')
print('-----', '-'*80)
for i, topic_dist in enumerate(topic_word):
    topic_words = np.array(vocab)[np.argsort(topic_dist)][:-(n_top_words+1):-1]
    top_words = ' '.join(topic_words)
    topic_summaries.append(top_words)
    print('  %3d %s' % (i, top_words))

Group Top Words
----- --------------------------------------------------------------------------------
    0 card software canon support windows player install cards works drivers
    1 fans joe huh chandler pink poster hence struggled chicken herb
    2 thin reality errors food team shower water rod cooking pratchett
    3 my product after i use size months them 3 these
    4 lhasa byron neil clippers austrian diversity luv ner plough nichols
    5 cost thanks business letter costs scarlet hot robert adventure england
    6 ear fit comedy 3d jawbone cheaper camcorder sandler adam named
    7 of the book in and is a his s read
    8 tribute ruth babe sincerely creamer rem dolls layer bela belkin
    9 von forces snmp cali perverse sanders ol diagnosed wagner armed
   10 ray blu dvd flavor smoke lake relative spider angles slasher
   11 fi sci la de l y jack force en 2000
   12 ice mount lcd jams ballet chase corner discs instruments celtic
   13 cd music album songs song s her is sound

## Modelling

Run the following cells to train a number of models on the count vector and TF-IDF vector feature sets generated above.

In [40]:
## helper function

def train_model(classifier, feature_vector_train, label, feature_vector_valid):
    # fit the training dataset on the classifier
    classifier.fit(feature_vector_train, label)

    # predict the labels on validation dataset
    predictions = classifier.predict(feature_vector_valid)

    return accuracy_score(predictions, y_test)

In [41]:
# Keep the results in a dataframe
results = pd.DataFrame(columns = ['Count Vectors',
                                  'WordLevel TF-IDF',
                                  'N-Gram Vectors',
                                  'CharLevel Vectors'])

### Naive Bayes Classifier

In [42]:
%%time
# Naive Bayes on Count Vectors
accuracy1 = train_model(MultinomialNB(), X_train_count, y_train, X_test_count)
print('NB, Count Vectors    : %.4f\n' % accuracy1)

NB, Count Vectors    : 0.8290

CPU times: user 8.86 ms, sys: 999 µs, total: 9.86 ms
Wall time: 9.84 ms


In [43]:
%%time
# Naive Bayes on Word Level TF IDF Vectors
accuracy2 = train_model(MultinomialNB(), X_train_tfidf, y_train, X_test_tfidf)
print('NB, WordLevel TF-IDF : %.4f\n' % accuracy2)

NB, WordLevel TF-IDF : 0.8370

CPU times: user 7.65 ms, sys: 997 µs, total: 8.65 ms
Wall time: 8.39 ms


In [44]:
%%time
# Naive Bayes on Ngram Level TF IDF Vectors
accuracy3 = train_model(MultinomialNB(), X_train_tfidf_ngram, y_train, X_test_tfidf_ngram)
print('NB, N-Gram Vectors   : %.4f\n' % accuracy3)

NB, N-Gram Vectors   : 0.8335

CPU times: user 9.01 ms, sys: 0 ns, total: 9.01 ms
Wall time: 9.07 ms


In [45]:
%%time
# # Naive Bayes on Character Level TF IDF Vectors
accuracy4 = train_model(MultinomialNB(), X_train_tfidf_ngram_chars, y_train, X_test_tfidf_ngram_chars)
print('NB, CharLevel Vectors: %.4f\n' % accuracy4)

NB, CharLevel Vectors: 0.8085

CPU times: user 35.6 ms, sys: 0 ns, total: 35.6 ms
Wall time: 34.9 ms


In [46]:
results.loc['Naïve Bayes'] = {
    'Count Vectors': accuracy1,
    'WordLevel TF-IDF': accuracy2,
    'N-Gram Vectors': accuracy3,
    'CharLevel Vectors': accuracy4}

### Linear Classifier

In [47]:
%%time
# Linear Classifier on Count Vectors
accuracy1 = train_model(LogisticRegression(solver = 'lbfgs', max_iter = 350), X_train_count, y_train, X_test_count)
print('LR, Count Vectors    : %.4f\n' % accuracy1)

LR, Count Vectors    : 0.8665

CPU times: user 9.88 s, sys: 26.3 ms, total: 9.91 s
Wall time: 6.51 s


In [48]:
%%time
# Linear Classifier on Word Level TF IDF Vectors
accuracy2 = train_model(LogisticRegression(solver = 'lbfgs', max_iter = 100), X_train_tfidf, y_train, X_test_tfidf)
print('LR, WordLevel TF-IDF : %.4f\n' % accuracy2)

LR, WordLevel TF-IDF : 0.8640

CPU times: user 158 ms, sys: 0 ns, total: 158 ms
Wall time: 92.9 ms


In [49]:
%%time
# Linear Classifier on Ngram Level TF IDF Vectors
accuracy3 = train_model(LogisticRegression(solver = 'lbfgs', max_iter = 100), X_train_tfidf_ngram, y_train, X_test_tfidf_ngram)
print('LR, N-Gram Vectors   : %.4f\n' % accuracy3)

LR, N-Gram Vectors   : 0.8300

CPU times: user 67 ms, sys: 1e+03 ns, total: 67 ms
Wall time: 33.2 ms


In [50]:
%%time
# Linear Classifier on Character Level TF IDF Vectors
accuracy4 = train_model(LogisticRegression(solver = 'lbfgs', max_iter = 100), X_train_tfidf_ngram_chars, y_train, X_test_tfidf_ngram_chars)
print('LR, CharLevel Vectors: %.4f\n' % accuracy4)

LR, CharLevel Vectors: 0.8360

CPU times: user 451 ms, sys: 2.96 ms, total: 454 ms
Wall time: 236 ms


In [51]:
results.loc['Logistic Regression'] = {
    'Count Vectors': accuracy1,
    'WordLevel TF-IDF': accuracy2,
    'N-Gram Vectors': accuracy3,
    'CharLevel Vectors': accuracy4}

### Support Vector Machine

In [52]:
%%time
# Support Vector Machine on Count Vectors
accuracy1 = train_model(LinearSVC(), X_train_count, y_train, X_test_count)
print('SVM, Count Vectors    : %.4f\n' % accuracy1)

SVM, Count Vectors    : 0.8370

CPU times: user 907 ms, sys: 4.97 ms, total: 912 ms
Wall time: 872 ms


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [53]:
%%time
# Support Vector Machine on Word Level TF IDF Vectors
accuracy2 = train_model(LinearSVC(), X_train_tfidf, y_train, X_test_tfidf)
print('SVM, WordLevel TF-IDF : %.4f\n' % accuracy2)

SVM, WordLevel TF-IDF : 0.8645

CPU times: user 92.3 ms, sys: 2 ms, total: 94.3 ms
Wall time: 93.6 ms


In [54]:
%%time
# Support Vector Machine on Ngram Level TF IDF Vectors
accuracy3 = train_model(LinearSVC(), X_train_tfidf_ngram, y_train, X_test_tfidf_ngram)
print('SVM, N-Gram Vectors   : %.4f\n' % accuracy3)

SVM, N-Gram Vectors   : 0.8115

CPU times: user 53.8 ms, sys: 0 ns, total: 53.8 ms
Wall time: 53.8 ms


In [55]:
%%time
# Support Vector Machine on Character Level TF IDF Vectors
accuracy4 = train_model(LinearSVC(), X_train_tfidf_ngram_chars, y_train, X_test_tfidf_ngram_chars)
print('SVM, CharLevel Vectors: %.4f\n' % accuracy4)

SVM, CharLevel Vectors: 0.8475

CPU times: user 952 ms, sys: 38 ms, total: 990 ms
Wall time: 989 ms


In [56]:
results.loc['Support Vector Machine'] = {
    'Count Vectors': accuracy1,
    'WordLevel TF-IDF': accuracy2,
    'N-Gram Vectors': accuracy3,
    'CharLevel Vectors': accuracy4}

### Bagging Models

In [57]:
%%time
# Bagging (Random Forest) on Count Vectors
accuracy1 = train_model(RandomForestClassifier(n_estimators = 100), X_train_count, y_train, X_test_count)
print('RF, Count Vectors    : %.4f\n' % accuracy1)

RF, Count Vectors    : 0.8210

CPU times: user 13.9 s, sys: 21.8 ms, total: 13.9 s
Wall time: 14.2 s


In [58]:
%%time
# Bagging (Random Forest) on Word Level TF IDF Vectors
accuracy2 = train_model(RandomForestClassifier(n_estimators = 100), X_train_tfidf, y_train, X_test_tfidf)
print('RF, WordLevel TF-IDF : %.4f\n' % accuracy2)

RF, WordLevel TF-IDF : 0.8290

CPU times: user 9.48 s, sys: 9.96 ms, total: 9.49 s
Wall time: 9.51 s


In [59]:
%%time
# Bagging (Random Forest) on Ngram Level TF IDF Vectors
accuracy3 = train_model(RandomForestClassifier(n_estimators = 100), X_train_tfidf_ngram, y_train, X_test_tfidf_ngram)
print('RF, N-Gram Vectors   : %.4f\n' % accuracy3)

RF, N-Gram Vectors   : 0.7780

CPU times: user 9.25 s, sys: 10.9 ms, total: 9.26 s
Wall time: 9.96 s


In [60]:
%%time
# Bagging (Random Forest) on Character Level TF IDF Vectors
accuracy4 = train_model(RandomForestClassifier(n_estimators = 100), X_train_tfidf_ngram_chars, y_train, X_test_tfidf_ngram_chars)
print('RF, CharLevel Vectors: %.4f\n' % accuracy4)

RF, CharLevel Vectors: 0.7810

CPU times: user 30.4 s, sys: 37.6 ms, total: 30.4 s
Wall time: 31.2 s


In [61]:
results.loc['Random Forest'] = {
    'Count Vectors': accuracy1,
    'WordLevel TF-IDF': accuracy2,
    'N-Gram Vectors': accuracy3,
    'CharLevel Vectors': accuracy4}

### Boosting Models

In [62]:
%%time
# Gradient Boosting on Count Vectors
accuracy1 = train_model(GradientBoostingClassifier(), X_train_count, y_train, X_test_count)
print('GB, Count Vectors    : %.4f\n' % accuracy1)

GB, Count Vectors    : 0.7950

CPU times: user 9.02 s, sys: 11.9 ms, total: 9.03 s
Wall time: 9.07 s


In [63]:
%%time
# Gradient Boosting on Word Level TF IDF Vectors
accuracy2 = train_model(GradientBoostingClassifier(), X_train_tfidf, y_train, X_test_tfidf)
print('GB, WordLevel TF-IDF : %.4f\n' % accuracy2)

GB, WordLevel TF-IDF : 0.7950

CPU times: user 20.9 s, sys: 15.9 ms, total: 20.9 s
Wall time: 20.9 s


In [64]:
%%time
# Gradient Boosting on Ngram Level TF IDF Vectors
accuracy3 = train_model(GradientBoostingClassifier(), X_train_tfidf_ngram, y_train, X_test_tfidf_ngram)
print('GB, N-Gram Vectors   : %.4f\n' % accuracy3)

GB, N-Gram Vectors   : 0.7405

CPU times: user 13.2 s, sys: 11.9 ms, total: 13.2 s
Wall time: 13.2 s


In [65]:
%%time
# Gradient Boosting on Character Level TF IDF Vectors
accuracy4 = train_model(GradientBoostingClassifier(), X_train_tfidf_ngram_chars, y_train, X_test_tfidf_ngram_chars)
print('GB, CharLevel Vectors: %.4f\n' % accuracy4)

GB, CharLevel Vectors: 0.7990

CPU times: user 3min 11s, sys: 155 ms, total: 3min 11s
Wall time: 3min 11s


In [66]:
results.loc['Gradient Boosting'] = {
    'Count Vectors': accuracy1,
    'WordLevel TF-IDF': accuracy2,
    'N-Gram Vectors': accuracy3,
    'CharLevel Vectors': accuracy4}

In [67]:
results

,Count Vectors,WordLevel TF-IDF,N-Gram Vectors,CharLevel Vectors
Naïve Bayes,0.8290,0.8370,0.8335,0.8085
Logistic Regression,0.8665,0.8640,0.8300,0.8360
Support Vector Machine,0.8370,0.8645,0.8115,0.8475
Random Forest,0.8210,0.8290,0.7780,0.7810
Gradient Boosting,0.7950,0.7950,0.7405,0.7990


Which combination of features and model performed the best?

Support Vector Machine + Word-Level TF-IDF performed the best (0.8645 accuracy), followed by Logistic Regression + Word-Level TF-IDF (0.8640 accuracy)



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



